# LSTM-arithmetic



In [1]:
# ! pip install seaborn
# ! pip install opencc
# ! pip install -U scikit-learn

import numpy as np
import pandas as pd
import torch
import torch.nn
import torch.nn.utils.rnn
import torch.utils.data
import matplotlib.pyplot as plt
import seaborn as sns
import opencc
import os
from sklearn.model_selection import train_test_split

data_path = './data'

In [2]:
df_train = pd.read_csv(os.path.join(data_path, 'arithmetic_train.csv'))
df_eval = pd.read_csv(os.path.join(data_path, 'arithmetic_eval.csv'))
df_train.head()

,src,tgt
0,(40*0)-18=,-18
1,6+4+31=,41
2,4*11+13=,57
3,44*39*26=,44616
4,(39*42)+27=,1665


In [3]:
# transform the input data to string
df_train['tgt'] = df_train['tgt'].apply(lambda x: str(x))
df_train['src'] = df_train['src'].add(df_train['tgt'])
df_train['len'] = df_train['src'].apply(lambda x: len(x))

df_eval['tgt'] = df_eval['tgt'].apply(lambda x: str(x))

# Build Dictionary
 - The model cannot perform calculations directly with plain text.
 - Convert all text (numbers/symbols) into numerical representations.
 - Special tokens
    - '&lt;pad&gt;'
        - Each sentence within a batch may have different lengths.
        - The length is padded with '&lt;pad&gt;' to match the longest sentence in the batch.
    - '&lt;eos&gt;'
        - Specifies the end of the generated sequence.
        - Without '&lt;eos&gt;', the model will not know when to stop generating.

In [4]:
char_to_id = {}
id_to_char = {}

# write your code here
# Build a dictionary and give every token in the train dataset an id
# The dictionary should contain <eos> and <pad>
# char_to_id is to conver charactors to ids, while id_to_char is the opposite

char_set = set()
for expr in df_train["src"]:
    for c in expr:
        char_set.add(c)

char_list = ["<pad>", "<eos>"] + sorted(list(char_set))

char_to_id = {c: id for id, c in enumerate(char_list)}
id_to_char = {id: c for id, c in enumerate(char_list)}

vocab_size = len(char_to_id)
print('Vocab size: {}'.format(vocab_size))

Vocab size: 18


# Data Preprocessing
 - The data is processed into the format required for the model's input and output. (End with \<eos\> token)


In [5]:
# Write your code here

all_char_id_lists = []
all_label_id_lists = []

for expr in df_train["src"]:
    # Build "char_id_list" column
    char_id_list = [char_to_id[c] for c in expr]
    all_char_id_lists.append(char_id_list)

    # Build "label_id_list" column
    label_id_list = []
    flag = False
    for c in expr:
        if c == "=":
            flag = True
            continue
        label_id_list.append(char_to_id[c]) if flag else label_id_list.append(char_to_id["<pad>"])
        

    all_label_id_lists.append(label_id_list + [char_to_id['<eos>']])

df_train["char_id_list"] = all_char_id_lists
df_train["label_id_list"] = all_label_id_lists

df_train.head()

,src,tgt,len,char_id_list,label_id_list
0,(40*0)-18=-18,-18,13,"[2, 11, 7, 4, 7, 3, 6, 8, 15, 17, 6, 8, 15]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 6, 8, 15, 1]"
1,6+4+31=41,41,9,"[13, 5, 11, 5, 10, 8, 17, 11, 8]","[0, 0, 0, 0, 0, 0, 11, 8, 1]"
2,4*11+13=57,57,10,"[11, 4, 8, 8, 5, 8, 10, 17, 12, 14]","[0, 0, 0, 0, 0, 0, 0, 12, 14, 1]"
3,44*39*26=44616,44616,14,"[11, 11, 4, 10, 16, 4, 9, 13, 17, 11, 11, 13, ...","[0, 0, 0, 0, 0, 0, 0, 0, 11, 11, 13, 8, 13, 1]"
4,(39*42)+27=1665,1665,15,"[2, 10, 16, 4, 11, 9, 3, 5, 9, 14, 17, 8, 13, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 13, 13, 12, 1]"


# Hyper Parameters

|Hyperparameter|Meaning|Value|
|-|-|-|
|`batch_size`|Number of data samples in a single batch|64|
|`epochs`|Total number of epochs to train|10|
|`embed_dim`|Dimension of the word embeddings|256|
|`hidden_dim`|Dimension of the hidden state in each timestep of the LSTM|256|
|`lr`|Learning Rate|0.001|
|`grad_clip`|To prevent gradient explosion in RNNs, restrict the gradient range|1|

In [6]:
batch_size = 64
epochs = 100
embed_dim = 128
hidden_dim = 512
lr = 0.0001
grad_clip = 1

# Data Batching
- Use `torch.utils.data.Dataset` to create a data generation tool called  `dataset`.
- The, use `torch.utils.data.DataLoader` to randomly sample from the `dataset` and group the samples into batches.

- Example: 1+2-3=0
    - Model input: 1 + 2 - 3 = 0
    - Model output: / / / / / 0 &lt;eos&gt;  (the '/' can be replaced with &lt;pad&gt;)
    - The key for the model's output is that the model does not need to predict the next character of the previous part. What matters is that once the model sees '=', it should start generating the answer, which is '0'. After generating the answer, it should also generate&lt;eos&gt;

In [7]:
# For training set
class Dataset(torch.utils.data.Dataset):
    def __init__(self, sequences):
        self.sequences = sequences
    
    def __len__(self):
        # return the amount of data
        return len(self.sequences["char_id_list"]) # Write your code here
    
    def __getitem__(self, index):
        # Extract the input data x and the ground truth y from the data
        x = self.sequences["char_id_list"][index] # Write your code here
        y = self.sequences["label_id_list"][index] # Write your code here
        return x, y

# collate function, used to build dataloader
def collate_fn(batch):
    batch_x = [torch.tensor(data[0]) for data in batch]
    batch_y = [torch.tensor(data[1]) for data in batch]
    batch_x_lens = torch.LongTensor([len(x) for x in batch_x])
    batch_y_lens = torch.LongTensor([len(y) for y in batch_y])
    
    # Pad the input sequence
    pad_batch_x = torch.nn.utils.rnn.pad_sequence(
        batch_x,
        batch_first=True,
        padding_value=char_to_id['<pad>']
    )
    
    pad_batch_y = torch.nn.utils.rnn.pad_sequence(
        batch_y,
        batch_first=True,
        padding_value=char_to_id['<pad>']
    )
    
    return pad_batch_x, pad_batch_y, batch_x_lens, batch_y_lens

In [8]:
ds_train = Dataset(df_train[["char_id_list", "label_id_list"]])

In [9]:
# Build dataloader of train set and eval set, collate_fn is the collate function
 # Write your code here
dl_train = torch.utils.data.DataLoader(
    dataset=ds_train,
    batch_size=batch_size,
    collate_fn=collate_fn,
)

# Model Design

## Execution Flow
1. Convert all characters in the sentence into embeddings.
2. Pass the embeddings through an LSTM sequentially.
3. The output of the LSTM is passed into another LSTM, and additional layers can be added.
4. The output from all time steps of the final LSTM is passed through a Fully Connected layer.
5. The character corresponding to the maximum value across all output dimensions is selected as the next character.

## Loss Function
Since this is a classification task, Cross Entropy is used as the loss function.

## Gradient Update
Adam algorithm is used for gradient updates.

In [10]:
class CharRNN(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super(CharRNN, self).__init__()
        # Write your code here
        # Design your own model, which should contain embedding layer, several hidden layers and a linear layer

        self.embedding = torch.nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_dim,
            padding_idx=char_to_id['<pad>']
        )
        self.lstm_layer = torch.nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            batch_first=True,
            num_layers=2
        )
        self.linear = torch.nn.Sequential(
            torch.nn.Linear(in_features=hidden_dim, out_features=hidden_dim),
            torch.nn.GELU(),
            torch.nn.Linear(in_features=hidden_dim, out_features=vocab_size)
        )
        
    def forward(self, batch_x, batch_x_lens):
        return self.encoder(batch_x, batch_x_lens)
    
    # The forward pass of the model
    def encoder(self, batch_x, batch_x_lens):
        batch_x = self.embedding(batch_x)
        batch_x = torch.nn.utils.rnn.pack_padded_sequence(
            batch_x,
            batch_x_lens,
            batch_first=True,
            enforce_sorted=False
        )
        batch_x, _ = self.lstm_layer(batch_x)
        batch_x, _ = torch.nn.utils.rnn.pad_packed_sequence(
            batch_x,
            batch_first=True
        )  
        batch_x = self.linear(batch_x)
       
        return batch_x

    def generator(self, start_char, max_len=200):
        # AI help: 
        # 1. understanding the meaning of each variable 
        # 2. unsqueeze() 
        # 3. how to parse y
        char_list = [char_to_id[c] for c in start_char]
        next_char = None
        while len(char_list) < max_len: 
            # Write your code here 
            # Pack the char_list to tensor
            # Input the tensor to the embedding layer, RNN layers, linear respectively
            device = next(self.parameters()).device
            input_tensor = torch.tensor(char_list, device=device).unsqueeze(0)
            input_lens = torch.tensor([len(char_list)])

            y = self.forward(input_tensor, input_lens) # Obtain the next token prediction y
            last_logits = y[0, -1, :]
            next_char = torch.argmax(last_logits, dim=-1).item() # Use argmax function to get the next token prediction
            
            if next_char == char_to_id['<eos>']:
                break
            
            char_list.append(next_char)
            
        return [id_to_char[ch_id] for ch_id in char_list]

In [11]:
torch.manual_seed(2)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu') # Write your code here. Specify a device (cuda or cpu)
model = CharRNN(
    vocab_size,
    embed_dim,
    hidden_dim
)

In [12]:
criterion = torch.nn.CrossEntropyLoss(ignore_index=char_to_id['<pad>']) # Write your code here. Cross-entropy loss function. The loss function should ignore <pad>
optimizer = torch.optim.Adam(model.parameters(), lr=lr) # Write your code here. Use Adam or AdamW for Optimizer

# Training
1. The outer `for` loop controls the `epoch`
    1. The inner `for` loop uses `data_loader` to retrieve batches.
        1. Pass the batch to the `model` for training.
        2. Compare the predicted results `batch_pred_y` with the true labels `batch_y` using Cross Entropy to calculate the loss `loss`
        3. Use `loss.backward` to automatically compute the gradients.
        4. Use `torch.nn.utils.clip_grad_value_` to limit the gradient values between `-grad_clip` &lt; and &lt; `grad_clip`.
        5. Use `optimizer.step()` to update the model (backpropagation).
2.  After every `1000` batches, output the current loss to monitor whether it is converging.

In [13]:
from tqdm import tqdm
from copy import deepcopy
model = model.to(device)
i = 0
for epoch in range(1, epochs+1):
    # The process bar
    model.train()
    bar = tqdm(dl_train, desc=f"Train epoch {epoch}")
    for batch_x, batch_y, batch_x_lens, batch_y_lens in bar:
        # Write your code here
        # Send all data into GPU
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)
        
        # Clear the gradient
        optimizer.zero_grad()

        # Prediction
        batch_pred_y = model(batch_x, batch_x_lens)
        
        # Write your code here
        # Input the prediction and ground truths to loss function
        # AI help: 
        # use view(...) to compute the Cross-entropy loss function 
        loss = criterion(batch_pred_y.view(-1, vocab_size), batch_y.view(-1))
        
        # Back propagation
        loss.backward()
        torch.nn.utils.clip_grad_value_(model.parameters(), grad_clip) # gradient clipping

        # Write your code here
        # Optimize parameters in the model
        optimizer.step()

        i += 1
        if i % 1000 == 0:
            bar.set_postfix(loss=loss.item())
    
    # Evaluate your model
    matched = 0
    total = 0
    bar_eval = tqdm(df_eval.iterrows(), desc=f"Validation epoch {epoch}")
    with torch.no_grad():
        for _, row in bar_eval:
            batch_x = row['src']
            batch_y = row['tgt']
            
            # prediction = model.generator("1+1=") # An example of using generator: model.generator('1+1=')
            
            # Write your code here. Input the batch_x to the model and generate the predictions
            # Write your code here.
            # Check whether the prediction match the ground truths
            # Compute exact match (EM) on the eval dataset
            # EM = correct/total

            prediction = model.generator(batch_x) # return ["1", "+", "1", "=", "2"]
            if "".join(prediction) == batch_x + batch_y:
                matched += 1

            total += 1

    print(matched / total)

# Save checkpoint.pt
torch.save(model.state_dict(), "checkpoint.pt")

Train epoch 1: 100%|██████████| 15625/15625 [05:20<00:00, 48.68it/s, loss=0.505]
Validation epoch 1: 10000it [01:47, 92.94it/s]


0.4333


Train epoch 2: 100%|██████████| 15625/15625 [02:10<00:00, 119.90it/s, loss=0.511]
Validation epoch 2: 10000it [00:50, 197.09it/s]


0.5341


Train epoch 3: 100%|██████████| 15625/15625 [02:54<00:00, 89.76it/s, loss=0.542] 
Validation epoch 3: 10000it [00:50, 198.45it/s]


0.5338


... (190 outputs left out)

Train epoch 99: 100%|██████████| 15625/15625 [02:04<00:00, 125.94it/s, loss=0.00147] 
Validation epoch 99: 10000it [00:49, 202.30it/s]


0.9868


Train epoch 100: 100%|██████████| 15625/15625 [02:08<00:00, 121.96it/s, loss=0.000194]
Validation epoch 100: 10000it [00:47, 211.05it/s]

0.9816


Running environment: 
- System: Windows 11
- CPU: 12th Gen Intel(R) Core(TM) i5-12600KF (3.70 GHz)
- GPU: NVIDIA GeForce RTX 4060 (8 GB)

Python version: 3.12.8

##### 1. Describe your model architecture, data preprocessing methods, and hyperparameter settings, and justify your choices.  (5%)

Answer: The model architecture consists of 1 Embedding layer, 2 LSTM layers, and 1 Linear layer using GELU as the activation function. Since LSTM achieved the best performance in Question 2, we selected it for our hidden layers. We also chose GELU because it is a widely adopted standard in modern architectures. There are 6 primary hyperparameters in our experiments: batch_size, epochs, embed_dim, hidden_dim, lr, and grad_clip. Through extensive experimentation, we observed:

- Batch size (64, 128, 256): While smaller values generally help generalization, we selected 64 to learn more details.
- Epochs (5, 10, 50, 100): Choosing a smaller learning rate requires a larger number of training epochs to reach full convergence.
- Learning rate (0.00005, 0.0001, 0.0005, 0.001): 0.0001 yields optimal stability, avoiding premature or unstable convergence.
- Embedding dimension (64, 128, 256): 128 offers the best performance, whereas 256 provides small gains at higher computational costs.
- Hidden dimension (128, 256, 512): Larger hidden dimensions generally improve network capacity and yield superior results.
- Gradient clipping: Set to 1 by default.

Therefore, we finalized our hyperparameter configurations as follows:

| Hyper-parameter | value  |
| --------------- | ------ |
| batch_size      | 128    |
| epochs          | 100    |
| lr              | 0.0001 |
| embed_dim       | 128    |
| hidden_dim      | 512    |
| grad_clip       | 1      |

The validation accuracy (Matched / Total) reaches 0.98 by epoch 57 and remains stably at 0.98 after epoch 80. The training loss converges to less than 0.001, demonstrating strong convergence. The experimental result is highly satisfactory.


##### 2. If you use 2-layer LSTM or GRU instead of RNN, what will happen to the quality of your answer generation? Why? (10%)

Answer: We fix the hyperparameters (batch_size = 64, epochs = 10, embed_dim = 128, hidden_dim = 512, lr = 0.0005, grad_clip = 1) and evaluate a network architecture with 1 Embedding layer, 2 Hidden layers, and 1 Linear layer with GELU as activation function across different architectures (RNN, LSTM, GRU). Each model requires approximately 2 minutes of training time per epoch. The validation accuracy (Matched / Total) for each model is shown below:

| epoch | RNN    | LSTM   | GRU    |
| ----- | ------ | ------ | ------ |
| 1     | 0.1515 | 0.5344 | 0.4953 |
| 2     | 0.1796 | 0.6998 | 0.5503 |
| 3     | 0.3278 | 0.7646 | 0.7102 |
| 4     | 0.3448 | 0.8697 | 0.7775 |
| 5     | 0.36   | 0.9172 | 0.7878 |
| 6     | 0.3765 | 0.9151 | 0.8131 |
| 7     | 0.3643 | 0.9147 | 0.8242 |
| 8     | 0.398  | 0.9308 | 0.8562 |
| 9     | 0.3861 | 0.9199 | 0.8845 |
| 10    | 0.4104 | 0.9458 | 0.8723 |

LSTM achieves the best performance (94.58%) because its cell state and three-gate mechanism (forget, input, output gates) effectively retain long-term memory while discarding irrelevant information, preventing vanishing gradients. GRU achieves 87.23%, which is similar to LSTM but with lower computational cost, since it uses only two gates (update and reset gates). RNN performs the worst (41.04%) due to vanishing gradients, which causes historical information to be lost.


##### 3. If we construct an training set using three-digit numbers while the evaluation set is constructed from two-digit numbers, what will happen to the quality of your answer generation? (10%)

Answer: We fix the model architecture and hyperparameters as Question 2. When trained only on three-digit numbers and evaluated on two-digit numbers, the generation quality degrades significantly, with validation accuracy dropping to 27.5% after 100 epochs. Three-digit number calculations require the LSTM hidden states to process three distinct magnitude steps (hundreds, tens, and units). When applied to two-digit number calculations as inputs, the learned hidden parameters fail to adjust to the shorter calculation sequence, causing positional misalignment and length bias—frequently over-generating digits or failing to emit the `<eos>` token at the correct step.

```
...
Train epoch 10: 100%|██████████| 7813/7813 [01:32<00:00, 84.50it/s, loss=0.916]
Validation epoch 10: 10000it [00:46, 215.92it/s]
0.0301
...
Train epoch 20: 100%|██████████| 7813/7813 [01:32<00:00, 84.66it/s, loss=0.756]
Validation epoch 20: 10000it [00:47, 211.68it/s]
0.067
...
Train epoch 30: 100%|██████████| 7813/7813 [01:32<00:00, 84.91it/s, loss=0.651]
Validation epoch 30: 10000it [00:47, 209.30it/s]
0.1286
...
Train epoch 40: 100%|██████████| 7813/7813 [01:32<00:00, 84.12it/s, loss=0.532]
Validation epoch 40: 10000it [00:48, 207.65it/s]
0.1555
...
Train epoch 50: 100%|██████████| 7813/7813 [01:33<00:00, 83.35it/s, loss=0.54] 
Validation epoch 50: 10000it [00:47, 210.53it/s]
0.1955
...
Train epoch 60: 100%|██████████| 7813/7813 [01:32<00:00, 84.25it/s, loss=0.538]
Validation epoch 60: 10000it [00:48, 208.01it/s]
0.2325
...
Train epoch 70: 100%|██████████| 7813/7813 [01:32<00:00, 84.59it/s, loss=0.478]
Validation epoch 70: 10000it [00:48, 208.05it/s]
0.2508
...
Train epoch 80: 100%|██████████| 7813/7813 [01:33<00:00, 83.26it/s, loss=0.56] 
Validation epoch 80: 10000it [00:48, 206.60it/s]
0.2529
...
Train epoch 90: 100%|██████████| 7813/7813 [01:33<00:00, 83.99it/s, loss=0.519]
Validation epoch 90: 10000it [00:49, 202.21it/s]
0.2505
...
Train epoch 100: 100%|██████████| 7813/7813 [01:32<00:00, 84.13it/s, loss=0.559]
Validation epoch 100: 10000it [00:49, 202.19it/s]
0.2752

```

##### 4. If we construct a training set that includes 20% incorrect answers, how will this affect the quality of the generated responses? Present some examples. (10%)

Answer: We use the same model architecture and the same hyper-parameters in Question 2. We construct two training dataset from the origin data :

1. Plausible Noise Training Dataset: 20% of target answers were injected with plausible arithmetic errors (e.g., small numerical offsets $\pm -5 \sim 5$ or sign flips).
2. Random Noise Training Dataset: 20% of target answers were replaced with completely random integers using `random.randint(-100000, 100000)`.

The validation accuracy (Matched / Total) and loss for each dataset is shown below:

| Epoch | Plausible Noise Accuracy | Random Noise Accuracy | Plausible Noise Loss | Random Noise Loss |
| ----- | ------------------------ | --------------------- | -------------------- | ----------------- |
| 1     | 0.5204                   | 0.498                 | 0.481                | 1.24              |
| 2     | 0.6339                   | 0.5819                | 0.51                 | 0.938             |
| 3     | 0.769                    | 0.6343                | 0.38                 | 0.955             |
| 4     | 0.8361                   | 0.6849                | 0.292                | 1.07              |
| 5     | 0.8497                   | 0.7123                | 0.261                | 1.04              |
| 6     | 0.8696                   | 0.7553                | 0.317                | 0.798             |
| 7     | 0.8788                   | 0.8359                | 0.341                | 0.886             |
| 8     | 0.8985                   | 0.8568                | 0.248                | 0.621             |
| 9     | 0.8936                   | 0.8684                | 0.16                 | 1.03              |
| 10    | 0.9191                   | 0.8844                | 0.216                | 0.715             |

Data noise directly influences model convergence, but the noise structure leads to distinct behaviors. Plausible noise shares identical syntax and length distributions with true targets, preventing the model from isolating errors as outliers; this degrades the accuracy (91.91%) while preserving general sequence learning. In contrast, unstructured random noise introduces unlearnable high-entropy noise that causes loss value unstable and a further drop in validation accuracy (88.44%).


##### 5. Why do we need gradient clipping during training? (5%)

Answer: Gradient clipping is an essential technique to prevent the exploding gradient problem, which often occurs when training RNNs over long sequences. If the gradients become too large, they cause unstable parameter updates, resulting in numerical overflow (such as NaN or Inf loss values). In `torch.nn.utils.clip_grad_value_()` function, `clip_value` is assigned and restrict the gradient in the range `[-clip_value, clip_value]`. By restricting the maximum gradient step size, gradient clipping ensures that weight updates remain controlled, keeping the training process stable.


##### … Anything that can strengthen your report. (5%)

Answer: In Question 1, our accuracy is up to 0.98. We define the "error ratio" by $$\frac{|\text{pred ans} - \text{ground truth}|}{|\text{ground truth}|}$$. To analyze the distribution of model inaccuracies without distortion from extreme statistical anomalies, we applied a data-filtering step before computing the diagnostic statistics:

1. **Target Zero Exclusion**: Filtered out samples where `ground_truth = 0` to prevent division-by-zero errors.
2. **Outlier Removal**: Removed outliers where `error_ratio > 1`.

After removing these rare outliers, the refined statistics for the remaining bounded error cases are presented below:

| Statistical indicators   | Value    |
| ------------------------ | -------- |
| Maximum                  | 0.2      |
| Minimum                  | 2.65e-05 |
| Mean                     | 0.0340   |
| Median                   | 0.0238   |
| Standard deviation       | 0.0351   |
| Coefficient of Variation | 1.0310   |

The coefficient of variation ($\text{CV} \approx 1.03$) alongside $\text{Median} < \text{Mean}$ demonstrates that the vast majority of predictions are extremely close to the ground truth rather than random outputs.


##### Generative AI Usage Statement

Answer: All code generated with the assistance of Gemini has been commented with the `AI help` tag. I wrote the initial code base myself and used AI as a review tool to verify that the implementation aligned with the requirements. In particular, AI was very helpful in troubleshooting tensor shape transitions. For example, in CrossEntropyLoss, we need to flat the tensor to match the input format.